In [3]:
import re
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from mofdb_client import fetch

/opt/miniconda3/envs/IML/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------

# Pressure points present in hMOF GCMC data (bar)
CO2_PRESSURES = [0.01, 0.05, 0.1, 0.5, 2.5]
PRESSURE_TOL  = 1e-3   # absolute tolerance for pressure matching (bar)

def _col(p: float) -> str:
    """Column name for a CO2 pressure point, e.g. 'co2_mol_kg_0.05bar'."""
    return f"co2_mol_kg_{p}bar"


def _parse_topology(mofid: str | None) -> str | None:
    """
    Extract net topology from a MOF-ID string.
    MOF-ID format: '<smiles_fragment(s)> MOFid-v1.<topology>.<cat>'
    """
    if not mofid:
        return None
    m = re.search(r"MOFid-v1\.([^.\s]+)", mofid)
    return m.group(1) if m else None


def _parse_metal_node(mofkey: str | None) -> str | None:
    """
    Extract metal-node label from a MOF-key string.
    MOF-key format: '<Metal>.<InChIKey1>.<InChIKey2>...MOFkey-v1.<topology>'
    The first token before the first '.' is the metal symbol(s).
    """
    if not mofkey:
        return None
    return mofkey.split(".")[0]


def _extract_co2_uptake(mof) -> dict:
    """
    Return a dict {col_name: uptake_mol_kg} for all CO2 pressure points.
    Picks the 298 K pure-CO2 simulation isotherm; falls back to any CO2 isotherm.
    If multiple isotherms exist, prefers simulated (simin='True') ones.
    """
    uptake = {_col(p): np.nan for p in CO2_PRESSURES}

    # Filter to CO2-only isotherms at 298 K
    co2_isos = [
        iso for iso in mof.isotherms
        if len(iso.adsorbates) == 1
        and iso.adsorbates[0].name == "CarbonDioxide"
        and abs(iso.temperature - 298.0) < 1.0
        and iso.pressureUnits.lower() in ("bar",)
        and "mol/kg" in iso.adsorptionUnits.lower()
    ]

    # Prefer simulated isotherms
    sim_isos = [iso for iso in co2_isos if str(iso.simin).lower() == "true"]
    chosen = sim_isos[0] if sim_isos else (co2_isos[0] if co2_isos else None)

    if chosen is None:
        return uptake

    for pt in chosen.isotherm_data:
        for target_p in CO2_PRESSURES:
            if abs(pt.pressure - target_p) < PRESSURE_TOL:
                # total_adsorption is the sum; for pure-CO2 it equals species_data[0].adsorption
                uptake[_col(target_p)] = pt.total_adsorption
                break

    return uptake


def _mof_to_row(mof) -> dict:
    """Convert a Mof object into a flat dict suitable for a DataFrame row."""
    sa_m2g   = mof.surface_area_m2g
    sa_m2cm3 = mof.surface_area_m2cm3

    # Density: rho = SA[m²/cm³] / SA[m²/g]  (algebraically: g/cm³)
    if sa_m2g and sa_m2cm3 and sa_m2g > 0:
        density = sa_m2cm3 / sa_m2g
    else:
        density = np.nan

    row = {
        "name"               : mof.name,
        "mofdb_id"           : mof.id,
        "database"           : mof.database,
        "pld"                : mof.pld,
        "lcd"                : mof.lcd,
        "surface_area_m2g"   : sa_m2g,
        "surface_area_m2cm3" : sa_m2cm3,
        "void_fraction"      : mof.void_fraction,
        "density_g_cm3"      : density,
        "topology"           : _parse_topology(mof.mofid),
        "metal_node"         : _parse_metal_node(mof.mofkey),
        "elements"           : ",".join(str(e) for e in mof.elements),
        "mofid"              : mof.mofid,
        "mofkey"             : mof.mofkey,
    }
    row.update(_extract_co2_uptake(mof))
    return row

In [5]:
# ---------------------------------------------------------------------------
# Download with checkpointing
# ---------------------------------------------------------------------------

CHECKPOINT_PATH = Path("hmof_checkpoint.pkl")   # raw list of row dicts
PARQUET_PATH    = Path("hmof_dataset.parquet")  # final assembled DataFrame
CHECKPOINT_EVERY = 1000                          # save every N structures

def download_hmof(resume: bool = True) -> list[dict]:
    """
    Stream all hMOF structures from MOFDB and return a list of row dicts.
    Saves a checkpoint every CHECKPOINT_EVERY entries so the download can
    be resumed after interruption.
    """
    rows: list[dict] = []
    start_idx = 0

    if resume and CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH, "rb") as f:
            rows = pickle.load(f)
        start_idx = len(rows)
        print(f"Resuming from checkpoint: {start_idx} structures already downloaded.")

    with tqdm(desc="Downloading hMOF", unit=" MOF", initial=start_idx,
              miniters=100, dynamic_ncols=True) as pbar:
        for i, mof in enumerate(fetch(database="hMOF")):
            if i < start_idx:
                continue   # skip already-processed structures
            rows.append(_mof_to_row(mof))
            pbar.update(1)

            if len(rows) % CHECKPOINT_EVERY == 0:
                with open(CHECKPOINT_PATH, "wb") as f:
                    pickle.dump(rows, f)

    # Final save
    with open(CHECKPOINT_PATH, "wb") as f:
        pickle.dump(rows, f)

    print(f"\nDownload complete. Total structures: {len(rows)}")
    return rows

In [6]:
rows = download_hmof(resume=True)


Download complete. Total structures: 137953


In [ ]:
# ---------------------------------------------------------------------------
# Assemble DataFrame and save to Parquet
# ---------------------------------------------------------------------------

df = pd.DataFrame(rows)

# Enforce sensible dtypes
float_cols = ["pld", "lcd", "surface_area_m2g", "surface_area_m2cm3",
              "void_fraction", "density_g_cm3"] + [_col(p) for p in CO2_PRESSURES]
for c in float_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["mofdb_id"] = pd.to_numeric(df["mofdb_id"], errors="coerce").astype("int64")

# Reorder columns for readability
geo_cols = ["pld", "lcd", "surface_area_m2g", "surface_area_m2cm3",
            "void_fraction", "density_g_cm3"]
co2_cols = [_col(p) for p in CO2_PRESSURES]
meta_cols = ["topology", "metal_node", "elements", "mofid", "mofkey", "database", "mofdb_id"]

df = df[["name"] + geo_cols + co2_cols + meta_cols]

# Drop only rows where CO2 targets or topology are missing
drop_cols = co2_cols + ["topology"]
df = df.dropna(subset=drop_cols)

# Save to CSV first (parquet via subprocess to avoid pyarrow kernel conflict)
CSV_PATH = Path("hmof_dataset.csv")
df.to_csv(CSV_PATH, index=False)

# Convert CSV → Parquet in a clean subprocess (no kernel-level pyarrow conflict)
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-c",
     f"import pandas as pd; pd.read_csv('{CSV_PATH}').to_parquet('{PARQUET_PATH}', index=False)"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("Parquet conversion failed:", result.stderr)
else:
    CSV_PATH.unlink()  # remove intermediate CSV
    print(f"Saved → {PARQUET_PATH}  ({df.shape[0]:,} rows × {df.shape[1]} columns)")

df.head(3)

Saved → hmof_dataset.parquet  (117,148 rows × 19 columns)


,name,pld,lcd,surface_area_m2g,surface_area_m2cm3,void_fraction,density_g_cm3,co2_mol_kg_0.01bar,co2_mol_kg_0.05bar,co2_mol_kg_0.1bar,co2_mol_kg_0.5bar,co2_mol_kg_2.5bar,topology,metal_node,elements,mofid,mofkey,database,mofdb_id
0,hMOF-6,9.75,10.75,3174.5,2227.6,0.754903,0.701717,0.107653,0.702453,1.190170,2.801410,8.49693,pcu,Zn,"Zn,O,C,H,F",[O-]C(=O)c1cc(F)c(c(c1F)F)C(=O)[O-].[O-]C(=O)c...,Zn.LWKOWFVDUSZDRA.SEOCHRNHZMPVAE.YUWKPDBHJFNMA...,hMOF,15338
1,hMOF-0,10.75,11.75,3676.3,2262.5,0.795539,0.615429,0.022406,0.073552,0.225851,0.885221,4.33307,pcu,Zn,"Zn,O,C,H",[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn...,Zn.KKEYFWRCBNTPAC.MOFkey-v1.pcu,hMOF,15339
2,hMOF-7,2.25,4.25,422.4,592.9,0.327134,1.403646,1.203980,2.480160,3.162250,4.512620,5.46213,pcu,Zn,"Zn,O,C,H,F",[O-]C(=O)c1cc(F)c(c(c1F)F)C(=O)[O-].[O-]C(=O)c...,Zn.LWKOWFVDUSZDRA.SEOCHRNHZMPVAE.YUWKPDBHJFNMA...,hMOF,15340


In [15]:
# ---------------------------------------------------------------------------
# Quick sanity checks
# ---------------------------------------------------------------------------

print("=== Shape ===")
print(df.shape)

print("\n=== Dtypes ===")
print(df.dtypes)

print("\n=== Missing values (%) ===")
print((df.isnull().mean() * 100).round(2).to_string())

print("\n=== Descriptor statistics ===")
print(df[geo_cols + co2_cols].describe().round(3).to_string())

print(f"\n=== Unique topologies ({df['topology'].nunique()}) ===")
print(df["topology"].value_counts().head(20).to_string())

print(f"\n=== Unique metal nodes ({df['metal_node'].nunique()}) ===")
print(df["metal_node"].value_counts().head(20).to_string())

=== Shape ===
(117148, 19)

=== Dtypes ===
name                   object
pld                   float64
lcd                   float64
surface_area_m2g      float64
surface_area_m2cm3    float64
void_fraction         float64
density_g_cm3         float64
co2_mol_kg_0.01bar    float64
co2_mol_kg_0.05bar    float64
co2_mol_kg_0.1bar     float64
co2_mol_kg_0.5bar     float64
co2_mol_kg_2.5bar     float64
topology               object
metal_node             object
elements               object
mofid                  object
mofkey                 object
database               object
mofdb_id                int64
dtype: object

=== Missing values (%) ===
name                  0.0
pld                   0.0
lcd                   0.0
surface_area_m2g      0.0
surface_area_m2cm3    0.0
void_fraction         0.0
density_g_cm3         0.0
co2_mol_kg_0.01bar    0.0
co2_mol_kg_0.05bar    0.0
co2_mol_kg_0.1bar     0.0
co2_mol_kg_0.5bar     0.0
co2_mol_kg_2.5bar     0.0
topology              0.0
metal_n

**Descriptors extracted per structure:**
| Field | Description |
|---|---|
| `pld` | Pore-limiting diameter (Å) |
| `lcd` | Largest-cavity diameter (Å) |
| `surface_area_m2g` | Accessible surface area (m²/g) |
| `surface_area_m2cm3` | Accessible surface area (m²/cm³) |
| `void_fraction` | Helium void fraction / porosity |
| `density_g_cm3` | Crystal density (g/cm³), derived as SA[m²/cm³] / SA[m²/g] |
| `topology` | Net topology parsed from MOF-ID (e.g. `pcu`, `dia`) |
| `metal_node` | Metal symbol(s) parsed from MOF-key |
| `elements` | Full element list |